First OpenAI demo

In [ ]:
import os
from openai import OpenAI
 
os.environ["OPENAI_API_KEY"] = 
 
client = OpenAI()
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Give me a short definition of DNN."}
    ]
)
 
print(response.choices[0].message.content)
 

Context-Aware Prompt Engine Code

In [ ]:
from openai import OpenAI
import json
client = OpenAI()
class ContextAwarePromptEngine:
    
    def __init__(self, model_name="gpt-4o-mini"):
        self.model_name = model_name

    def build_prompt(self, user_query: str, max_budget: int):
        system_instruction = """
        You are an Enterprise AI Commerce Assistant.
        Follow all business rules strictly.
        Always return valid JSON only.
        Do not provide explanations outside JSON.
        """

        business_rules = f"""
        Business Constraints:
        - Recommend only running shoes.
        - Maximum price allowed: ₹{max_budget}.
        - Provide concise professional reasoning.
        """

        output_format = """
        Return strictly in this JSON format:
        {
        "recommendations": [
        {
        "name": "Product Name",
        "price": 0000,
        "reason": "Professional justification"
        }
        ]
        }
        """

        final_prompt = f"""
        {system_instruction}
        {business_rules}
        {output_format}
        User Query:
        {user_query}
        """
        return final_prompt

    def generate(self, user_query: str, max_budget: int):
        prompt = self.build_prompt(user_query, max_budget)
        response = client.chat.completions.create(
        model=self.model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
        )
        return response.choices[0].message.content
        
engine = ContextAwarePromptEngine()
result = engine.generate(
user_query="Suggest running shoes suitable for daily jogging.",
max_budget=5000
)
print(result)

{
  "recommendations": [
    {
      "name": "Nike Revolution 5",
      "price": 4495,
      "reason": "Lightweight and breathable, ideal for daily jogging."
    },
    {
      "name": "Adidas Duramo 9",
      "price": 4999,
      "reason": "Comfortable cushioning and support for long runs."
    },
    {
      "name": "Puma Ignite Flash Evoknit",
      "price": 4990,
      "reason": "Offers great energy return and stability for jogging."
    },
    {
      "name": "Asics Gel-Excite 7",
      "price": 4999,
      "reason": "Provides excellent shock absorption and comfort."
    }
  ]
}


In [15]:
from openai import OpenAI
import json
client = OpenAI()
class ContextAwarePromptEngine:
    
    def __init__(self, model_name="gpt-4o-mini"):
        self.model_name = model_name

    def build_prompt(self, product_name: str):
        system_instruction = """
        You are an Enterprise AI Commerce Assistant.
        Follow all business rules strictly.
        Always return valid JSON only.
        Do not provide explanations outside JSON.
        """

        business_rules = f"""
        Business Constraints:
        - Give me the {product_name} product's short and long description.
        - Provide concise professional reasoning.
        """

        output_format = """
        Return strictly in this JSON format:
        {
        "Descriptions": [
        {
        "Product name": "Product Name",
        "Short Description": "Short description",
        "Long Description": "Long description"
        }
        ]
        }
        """

        final_prompt = f"""
        {system_instruction}
        {business_rules}
        {output_format}
        User Query:
        {product_name}
        """
        return final_prompt

    def generate(self, product_name: str):
        prompt = self.build_prompt(product_name)
        response = client.chat.completions.create(
        model=self.model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
        )
        return response.choices[0].message.content
        
engine = ContextAwarePromptEngine()
result = engine.generate(
product_name="Nike Air Zoom Pegasus 38"
)
print(result)

{
    "Descriptions": [
        {
            "Product name": "Nike Air Zoom Pegasus 38",
            "Short Description": "The Nike Air Zoom Pegasus 38 is a versatile running shoe designed for comfort and performance, featuring responsive cushioning and a breathable upper.",
            "Long Description": "The Nike Air Zoom Pegasus 38 combines a lightweight design with responsive Zoom Air cushioning, making it ideal for both casual runners and serious athletes. The shoe features a breathable mesh upper for enhanced ventilation, while the padded collar and tongue provide additional comfort. With a durable rubber outsole for traction and a variety of color options, the Pegasus 38 is perfect for daily training or casual wear."
        }
    ]
}


In [16]:
import numpy as np
from openai import OpenAI
 
# -----------------------------------
# Client Initialization
# -----------------------------------
 
client = OpenAI()
 
EMBEDDING_MODEL = "text-embedding-3-small"
GEN_MODEL = "gpt-4o-mini"
 
# -----------------------------------
# Sample Product Documents
# -----------------------------------
 
documents = [
    {
        "id": 1,
        "text": "Nike Air Zoom: Designed for marathon runners with responsive cushioning and lightweight build.",
        "brand": "Nike"
    },
    {
        "id": 2,
        "text": "Adidas Ultraboost: Ideal for long-distance running with energy return foam.",
        "brand": "Adidas"
    },
    {
        "id": 3,
        "text": "Puma Flex Run: Budget-friendly daily jogging shoe with breathable mesh.",
        "brand": "Puma"
    },
    {
        "id": 4,
        "text": "ASICS Gel Nimbus: Premium marathon shoe with superior shock absorption.",
        "brand": "ASICS"
    }
]
 
# -----------------------------------
# Embedding Utility
# -----------------------------------
 
def generate_embedding(text):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding
 
 
 
# -----------------------------------
# Precompute & Cache Document Embeddings
# -----------------------------------
 
for doc in documents:
    doc["embedding"] = generate_embedding(doc["text"])
    print(f"Doc ID: {doc['id']}")
    print(f"Text: {doc['text']}")
    print(f"Embedding (first 10 dims): {doc['embedding'][:10]}")
    print("-" * 60)
 
 
# -----------------------------------
# Similarity Function
# -----------------------------------
 
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
 
 
# -----------------------------------
# Retrieval Layer
# -----------------------------------
 
def retrieve_top_k(query, k=2, debug=False):
    query_embedding = generate_embedding(query)
 
    scored_docs = []
    for doc in documents:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored_docs.append((score, doc))
 
    scored_docs.sort(reverse=True, key=lambda x: x[0])
 
    if debug:
        print("\nSimilarity Scores:")
        for score, doc in scored_docs:
            print(f"{round(score,4)} -> {doc['text']}")
 
    top_docs = [doc for _, doc in scored_docs[:k]]
    return top_docs
 
 
# -----------------------------------
# Context Assembly
# -----------------------------------
 
def build_context(docs):
    return "\n\n".join([doc["text"] for doc in docs])
 
 
# -----------------------------------
# Grounded Generation
# -----------------------------------
 
def generate_grounded_answer(query, debug=False):
 
    top_docs = retrieve_top_k(query, debug=debug)
 
    context_block = build_context(top_docs)
 
    messages = [
        {
            "role": "system",
            "content": (
                "You are a product recommendation assistant. "
                "Recommend products strictly from the provided context. "
                "If no relevant product exists, say: "
                "'Information not available in provided context.'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context_block}
 
User Question:
{query}
 
If a relevant product is found, recommend it clearly.
"""
        }
    ]
 
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=messages,
        temperature=0.1
    )
 
    return response.choices[0].message.content
 
 
# -----------------------------------
# Demo Execution
# -----------------------------------
 
if __name__ == "__main__":
    query = "shoes Ideal for long-distance "
    result = generate_grounded_answer(query, debug=True)
 
    print("\nFinal Answer:")
    print(result)
 

Doc ID: 1
Text: Nike Air Zoom: Designed for marathon runners with responsive cushioning and lightweight build.
Embedding (first 10 dims): [0.029780996963381767, 0.07538742572069168, 0.010278548114001751, -0.016304297372698784, -0.048570845276117325, -0.009702767245471478, -0.013419690541923046, 0.014537048526108265, 0.024832699447870255, 0.018516208976507187]
------------------------------------------------------------
Doc ID: 2
Text: Adidas Ultraboost: Ideal for long-distance running with energy return foam.
Embedding (first 10 dims): [-0.00015415340021718293, 0.07082030922174454, 0.03453642874956131, 0.006680359598249197, -0.00814870372414589, 0.003158153733238578, 0.0021812799386680126, 0.009101307950913906, 0.0667429193854332, 0.026867059990763664]
------------------------------------------------------------
Doc ID: 3
Text: Puma Flex Run: Budget-friendly daily jogging shoe with breathable mesh.
Embedding (first 10 dims): [-0.027058783918619156, 0.0401313416659832, 0.014314214698970